<a id="unit-u.dc660ddafff6474285cb9d663e37771e"></a>

<a id="observability-introduction-start-the-dog-stack"></a>

# Жүйені бақылауға кіріспе: DOG стегін іске қосу

**1-модуль: DOG стегіндегі жүйені бақылау негіздері** · [Сабақ](course.md) · [Тест](quiz01_observability_foundations.ipynb)

Бұл зертханалық жұмыста DOG стегін (Doris, OpenTelemetry Collector және Grafana) және сервистері стекке телеметрияны үздіксіз жіберетін OpenTelemetry демо онлайн дүкенін іске қосасыз. Doris-тен телеметрияның жеткенін тексеріп, дүкен мен Grafana-ны көресіз. Содан кейін курстың қалған бөлігі тексеретін инцидентті бастайсыз: демоның мүмкіндік жалаушаларының бірімен дүкен төлемдерінің жартысын сәтсіз етіп, сәтсіз тапсырыстардың Doris-те пайда болуын бақылайсыз.

Шамамен 35 минут және алғашқы бейнелерді жүктеуге (шамамен 4.8 GB) тағы бірнеше минут жоспарлаңыз. Ұяшықтарды ретімен орындаңыз.

<a id="unit-u.9915b2a59a50414c8131da1fbd1b1c13"></a>

<a id="initialize-the-lab"></a>

### Зертханалық жұмысты инициализациялау

`lab.shell(...)` не `lab.sql(...)` қолданбас бұрын осы ұяшықты бір рет орындаңыз.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


<a id="unit-u.c36cb1763a7644e18d0157926a03a6db"></a>

<a id="1-start-the-dog-stack"></a>

## 1. DOG стегін іске қосу

Орта ұяшығы `.runtime/` астына DOG стегінің бекітілген нұсқасын алып, оның төрт контейнерін іске қосады: `doris-obs` (Doris), `doris-obs-otel-collector`, `doris-obs-grafana` және 8-модульдегі Grafana ескерту хаттарын қабылдайтын жергілікті пошта сервері `doris-obs-mailpit`. Сондай-ақ 2-модульде FE журналын жүктеу үшін Doris FE журнал каталогын осы компьютердегі `.runtime/doris-logs/fe/` жолына байланыстырады.

In [ ]:
lab.prepare_environment()


<a id="unit-u.5cc1b178a3614e1cbbfd9cbe9587496b"></a>

**Күтілетін нәтиже**

Хост порттарын тексеру жеті портты `available`, ал ұяшықты қайта орындағанда `used by this course` деп көрсетеді. Командалар журналы жұмыс істеп тұрған төрт контейнермен аяқталады. Doris түйіндерінің кестесінде бір FE мен бір BE бар, екеуі де тірі, нұсқасы `doris-4.1.3-rc02-7126cf65d96`; соңғы карточкада Grafana, Doris және Mailpit адрестері көрсетіледі. Алғашқы іске қосуда шамамен 2.1 GB бейне жүктеледі; қайталап орындау жұмыс істейтін контейнерлерді қайта пайдаланып, бірнеше секунд алады.

<a id="unit-u.317e6fa9208c4ac98a3331fac9d016fa"></a>

<a id="2-start-the-opentelemetry-demo"></a>

## 2. OpenTelemetry демосын іске қосу

OpenTelemetry демосы — OpenTelemetry құралдарымен жабдықталған оннан астам сервистен тұратын Astronomy Shop онлайн дүкені. Оның жүктеме генераторы дүкенді үздіксіз қарап, тапсырыс береді, сондықтан сервистер трасса, журнал және метрикаларды үздіксіз өндіреді. Демоның Collector-ы оларды DOG Collector-ына бағыттайды, ал ол Doris-ке жазады. Кейінгі модульдер осы деректерді талдайды.

Келесі ұяшық демоның 23 контейнерін іске қосады. Алғашқы іске қосуда шамамен 2.7 GB бейне жүктеліп, бірнеше минут кетеді. Содан кейін ұяшық әр контейнер іске қосылып, дүкен жауап бергенше күтеді.

In [ ]:
lab.start_demo()


<a id="unit-u.27c3173341c24c388f8aa0bc57175b23"></a>

**Күтілетін нәтиже**

Порт тексеруі `28082` портын `available`, ал ұяшықты қайта орындағанда `used by this course` деп көрсетеді. Командалар журналында `doris-obs-demo-*` деп аталатын 23 контейнер тізіледі. Ұяшық «The demo is running» карточкасымен аяқталады: барлық 23 контейнер жұмыс істеп, дүкен жауап береді; бұл әдетте жарты минут ішінде болады. Демо іске қосылғанда контейнер қайта іске қосылса, карточка үстіндегі кестеде көрсетіледі. Ол қайта жұмыс істеп тұр, ал келесі тексеру оның телеметриясы келетінін көрсетеді.

<a id="unit-u.a2e0327a10f44c3eb2c8352444762b5b"></a>

<a id="check-that-the-telemetry-reached-doris"></a>

### Телеметрияның Doris-ке жеткенін тексеру

Жұмыс істеп тұрған контейнерлер телеметрияның Doris-ке жеткенін көрсетпейді. DOG Collector әр сигналды `otel` дерекқорындағы жеке кестеге жазады: span-дарды `otel_traces`, журналдарды `otel_logs`, ал метрикаларды әр түрге жеке кестеге, мысалы санауыштарды `otel_metrics_sum` кестесіне. Келесі ұяшық әр сигнал үшін соңғы бес минуттағы жолдарды және оларды жіберген сервистерді санайды. Кестелер уақыт белгілерін UTC бойынша сақтайды, сондықтан сұрау оларды `UTC_TIMESTAMP()` мәнімен салыстырады. Демо сервистері бірінен кейін бірі іске қосылады; сол себепті ұяшық үш сигналда да жолдар болғанша сұрауды қайталайды.

In [ ]:
demo_telemetry = lab.wait_for_rows(
    """
    SELECT signal, source_table, services, recent_rows, latest_timestamp
    FROM (
        SELECT 'traces' AS signal, 'otel_traces' AS source_table,
               COUNT(DISTINCT service_name) AS services, COUNT(*) AS recent_rows,
               MAX(timestamp) AS latest_timestamp
        FROM otel_traces
        WHERE timestamp >= UTC_TIMESTAMP() - INTERVAL 5 MINUTE
        UNION ALL
        SELECT 'logs', 'otel_logs', COUNT(DISTINCT service_name), COUNT(*), MAX(timestamp)
        FROM otel_logs
        WHERE timestamp >= UTC_TIMESTAMP() - INTERVAL 5 MINUTE
        UNION ALL
        SELECT 'metrics', 'otel_metrics_sum', COUNT(DISTINCT service_name), COUNT(*), MAX(timestamp)
        FROM otel_metrics_sum
        WHERE timestamp >= UTC_TIMESTAMP() - INTERVAL 5 MINUTE
    ) AS recent
    WHERE recent_rows > 0
    ORDER BY signal
    """,
    title="Telemetry from the last five minutes",
    hint="Read the demo Collector's log with docker logs doris-obs-demo-otel-collector and the DOG Collector's "
         "log with docker logs doris-obs-otel-collector, then rerun this cell.",
    min_rows=3,
    timeout_seconds=180,
)

<a id="unit-u.7f6409f43615449bb75932c582d871b7"></a>

**Күтілетін нәтиже**

Әдетте демо іске қосылғаннан кейін бір минут ішінде `logs`, `metrics` және `traces` деген үш жол шығады. Демо сервистері іске қосылған сайын `services` өседі: бірнеше минуттан соң шамамен 18 сервис трасса жібереді (олардың бірі, `frontend-web`, браузерде орындалатын дүкен коды), шамамен 10-ы журнал, 20-сы метрика жібереді. Метрикаларға Collector-лардың өз метрикалары және сервис атауы бос болатын хост пен контейнер метрикалары да кіреді. `latest_timestamp` қазіргі UTC уақытынан бірнеше секунд қана алшақ болады.

<a id="unit-u.71359e8fffaf40e3a57df693a51cad51"></a>

<a id="3-open-the-storefront"></a>

## 3. Дүкенді ашу

Демоны — телескоптар мен керек-жарақтардың Astronomy Shop онлайн дүкенін көру үшін [http://127.0.0.1:28082](http://127.0.0.1:28082) адресін (немесе `DORIS_OBS_DEMO_PORT` портын) ашыңыз. **Go Shopping** басып, тауарды ашыңыз да, себетке қосыңыз. Басуларыңыз дүкен сервистеріне сұрау жібереді және жүктеме генераторымен қатар телеметрия өндіреді.

<a id="unit-u.b1e9afba062e40bc9e67f2fb737071a1"></a>

<a id="4-open-grafana"></a>

## 4. Grafana-ны ашу

Grafana демо телеметриясын Doris-тен оқиды. [http://127.0.0.1:23000/a/velodb-doris-app/discover](http://127.0.0.1:23000/a/velodb-doris-app/discover) адресін (немесе `DORIS_OBS_GRAFANA_PORT` портын) ашыңыз; 1-бөлімдегі орта ұяшығы да осы адресті шығарады. Ол жүйеге кірмей-ақ Doris телеметриясын іздеуге арналған Grafana плагині Doris App-тың **Discover** бетін ашады. Бет жоғарғы **Table** тізімінде аталған `otel_logs` кестесінен іздеп, сәйкес келген әр журнал жазбасын `timestamp`, `service_name`, `body` сияқты бағандарымен көрсетеді. Компьютердің уақыт белдеуіне қарай әзірге жазба көрінбеуі мүмкін: Doris App жергілікті уақытпен сүзеді, ал Doris уақытты UTC бойынша сақтайды. 3-модуль бетті түсіндіріп, уақыт аралығын орнатады және сол арқылы сәтсіз тапсырыстарды табады.

<a id="unit-u.73d4541c44f543939ceb2b6e99d4a6c2"></a>

<a id="5-start-the-incident"></a>

## 5. Инцидентті бастау

Демо мүмкіндік жалаушаларымен ақау сценарийлерін қоса алады; оларды басқа сервистерге flagd жалауша сервисі береді. Курс солардың бірін қарастырады: `paymentFailure` payment сервисін төлем сұрауларының бір бөлігін қабылдамайтын етеді, сондықтан кей тұтынушылар тапсырысын аяқтай алмайды. Оны қазір қосасыз. Келесі модульдер себебі туралы білгеніңізді пайдаланбай, кезекші инженер сияқты ақауды телеметриядан табады.

1. Демо жалаушаларының [http://127.0.0.1:28082/feature](http://127.0.0.1:28082/feature) бетін (немесе `DORIS_OBS_DEMO_PORT` портын) ашыңыз. Әр жалауша үшін мәндер тізімі бар жеке карточка көрсетіледі.
2. «Fail payment service charge requests n%» деген **paymentFailure** карточкасының тізімінен **50%** таңдаңыз.
3. Беттің жоғарғы жағындағы **save** басыңыз.

Бет жалаушаларды `.runtime/` астына алынған репозиторийдегі flagd оқитын `doris-opentelemetry-demo/src/flagd/demo.flagd.json` файлына сақтайды; flagd жаңа мәнді дереу береді. Келесі ұяшық браузердегі дүкен беттері сияқты дүкен проксиі арқылы flagd жалаушаларын сұрап, `paymentFailure` мәні `50%` болғанша күтеді.

In [ ]:
lab.wait_for_feature_flag("paymentFailure", "50%");

<a id="unit-u.2b87f4526465494a99a5423af60442df"></a>

**Күтілетін нәтиже**

Ұяшық алдымен сұрауын терминалда да орындауға болатын `curl` командасы ретінде, содан кейін бір жолды көрсетеді: `paymentFailure`, `50%` нұсқасы және `0.5` мәні. 0 мен 1 арасындағы кездейсоқ сан осы мәннен төмен болса, payment сервисі төлемді сәтсіз етеді. Ұяшықты **save** баспай тұрып орындасаңыз, үш минутқа дейін күтеді; жалаушаны сақтағанда ұяшық жалғасады.

<a id="unit-u.9bb15e97284b432b8f86f7016c15bbba"></a>

<a id="see-the-failed-orders-in-doris"></a>

### Doris-тен сәтсіз тапсырыстарды көру

Бір тапсырыс — бір checkout. Checkout сервисі әр тапсырысты `oteldemo.CheckoutService/PlaceOrder` атты span ретінде тіркейді; тапсырыс сәтсіз болса, span күйі `STATUS_CODE_ERROR` болады. Жүктеме генераторы минутына бір-екі тапсырыс қана береді, сондықтан келесі ұяшық алғашқы сәтсіз тапсырыс Doris-ке жеткенше, ең көбі он минут, сұрауын қайталайды. Күткен кезде күй әр он секунд сайын өтіп кеткен уақытты, қалған уақытты және сәйкес нәтиже жолдарының санын көрсетіп жаңарады. Тапсырыс табылса жасыл болады, ал шекке жетсе қалпына келтіру кеңесін көрсетеді.

In [ ]:
first_failure = lab.wait_for_rows(
    """
    SELECT MIN(timestamp) AS first_failed_order, COUNT(*) AS failed_orders
    FROM otel_traces
    WHERE service_name = 'checkout'
      AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
      AND status_code = 'STATUS_CODE_ERROR'
      AND timestamp >= UTC_TIMESTAMP() - INTERVAL 5 MINUTE
    HAVING COUNT(*) > 0
    """,
    title="Failed orders in the last five minutes",
    hint="No order has failed yet. Check that paymentFailure is 50% (the previous cell) and that the demo runs, "
         "then rerun this cell: the load generator places only a few orders a minute.",
    timeout_seconds=600,
)

<a id="unit-u.07e7eca4cdb94192b55dea1d343df5fc"></a>

**Күтілетін нәтиже**

Әдетте жалаушаны сақтағаннан кейін екі-үш минутта бір жол шығады: соңғы бес минуттағы алғашқы сәтсіз тапсырыс уақыты UTC бойынша және осы кезге дейінгі сәтсіз тапсырыстар саны.

Тапсырысты өзіңіз де сәтсіз ете аласыз. Дүкенде тауарды себетке қосып, себетті ашыңыз да, **Place Order** басыңыз. Тапсырыс өтсе, дүкен «Your order is complete!» көрсетеді. Төлем сәтсіз болса, бет себетте қалып, хабар көрсетпейді: тұтынушы тек ештеңе болмағанын көреді. Қайталаңыз: шамамен әр екінші тапсырыс сәтсіз болады.

<a id="unit-u.d6166ca56b644d59917604a855246288"></a>

Соңғы 15 минуттағы тапсырыстарды минут және нәтиже бойынша санаңыз:

In [ ]:
lab.sql("""
SELECT DATE_FORMAT(timestamp, '%H:%i') AS minute_utc,
       COUNT(*) AS orders,
       SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS failed_orders
FROM otel_traces
WHERE service_name = 'checkout'
  AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 15 MINUTE
GROUP BY minute_utc
ORDER BY minute_utc
""", title="Orders per minute in the last 15 minutes");

<a id="unit-u.0853f2ce14a541f9b9e55cc086b761ab"></a>

**Күтілетін нәтиже**

Тапсырыс берілген әр минутқа бір жол, әдетте 1–3 тапсырыс. Жалаушаны сақтағанға дейін `failed_orders` — 0; кейін тапсырыстардың шамамен жартысы сәтсіз болады. Тапсырыс аз болғандықтан, бір минутта ешқайсысы не барлығы сәтсіз болуы мүмкін: үлес бірнеше минутта теңеседі. Курстың қалған бөлігі осы белгіні тексереді. 2-модуль телеметрияның әр түрінен сәтсіз тапсырыстың дәлелін табады, 3-модуль ақаудың қайдан және неге басталғанын анықтайды, ал 2-деңгей тексеруді жеделдетіп, ол туралы ескертеді.

<a id="unit-u.8a79adb5f38a4655b615b1f599a7b2d5"></a>

<a id="6-exercise-find-the-services-that-report-errors"></a>

## 6. Жаттығу: қате хабарлайтын сервистерді табу

Сәтсіз тапсырыс бірнеше сервисте қате span-дарын қалдырады. Соңғы 10 минуттағы күйі `STATUS_CODE_ERROR` span-дарды сервис бойынша санап, қатесі ең көп сервистерді алдымен шығаратын сұрау жазыңыз.

In [ ]:
# Your code here

<a id="unit-u.12528cad4c2940bda36637aa5e9d1184"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған. Салыстыру үшін ашыңыз немесе орындаңыз.

In [ ]:
lab.sql("""
SELECT service_name, COUNT(*) AS error_spans
FROM otel_traces
WHERE status_code = 'STATUS_CODE_ERROR'
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 10 MINUTE
GROUP BY service_name
ORDER BY error_spans DESC
""", title="Error spans of the last 10 minutes, by service");

<a id="unit-u.c075841b071f4e079faf622450463073"></a>

**Күтілетін нәтиже**

Тапсырыс жолындағы сервистер: әр сәтсіз тапсырысқа `frontend` шамамен үш қате span-ын, `frontend-proxy` мен `checkout` шамамен екіден, `payment` пен `load-generator` бірден тіркейді. Бір сұраудың бірнеше span-ы, мысалы кіріс сұрау span-ы мен келесі сервиске өз шақыруының span-ы қатемен аяқталса, сервис бір тапсырысқа бірнеше қате span-ын тіркейді. Дүкенде өзіңіз берген тапсырыста `load-generator` span-ы болмайды. Басқа сұраулардың бірнеше қате span-ы да көрінуі мүмкін: демо іске қосылғаннан кейін `frontend` пен `payment` бұлттық метадерек сервисіне жете алмайды, ал `accounting` Kafka тақырыбы құрылмай тұрып оны оқи алмайды; кейін жүктеме генераторының браузері кей сурет жүктеулерін тоқтатады. Демек қате span-дары ақау басталған сервисті емес, оны хабарлаған барлық сервисті көрсетеді; 3-модуль сол сервисті табу үшін бір сәтсіз тапсырысты қадағалайды.

<a id="unit-u.7c95828f37a74c7594e37a5db3ae1ef1"></a>

<a id="7-stop-the-environment"></a>

## 7. Ортаны тоқтату

Демо 23 контейнерді іске қосып, кейінгі модульдер талдайтын телеметрияны үздіксіз жібереді. Сондықтан курс бойы оны қосулы қалдырыңыз. Жадыны босату қажет болса, `lab.stop_demo()` орындаңыз: Doris-тегі бұрынғы телеметрия қалады, ал `lab.start_demo()` демоны қайта іске қосады. Мүмкіндік жалаушалары сіз орнатқан күйде сақталады, сондықтан қайта іске қосқаннан кейін төлем ақауы жалғасады. Инцидентті аяқтау үшін жалауша бетінде **paymentFailure** мәнін **off** етіп қайтарыңыз.

Курсты аяқтағанда `lab.teardown()` курс контейнерлерін, желілері мен томдарын жояды және демоның мүмкіндік жалаушаларын өшіреді. [Курс README файлындағы](../../README.md) тоқтату мен тазалау бөлімін қараңыз.

<a id="unit-u.14e960e6c4714698b8a68efc7fda1301"></a>

<a id="you-are-done-when"></a>

## Аяқтау шарттары

- DOG стегінің төрт контейнері мен демоның 23 контейнері жұмыс істейді;
- `otel_traces`, `otel_logs` және `otel_metrics_sum` кестелерінде соңғы бес минуттың жолдары бар;
- `paymentFailure` мәні `50%`, ал Doris-те сәтсіз тапсырыстар — күйі `STATUS_CODE_ERROR` болатын `oteldemo.CheckoutService/PlaceOrder` span-дары бар.

<a id="unit-u.eacd0093f85f4c8cabc832fe0203033a"></a>

<a id="next"></a>

## Әрі қарай

[1-тест: DOG стегіндегі жүйені бақылау негіздері](quiz01_observability_foundations.ipynb) файлын ашып, [2-модуль: Жүйені бақылау деректерін жүктеу](../module02-data-ingest/course.md) бөліміне өтіңіз.